In [1]:
import sys

In [2]:
import gbtoolbox.bounds as bounds
import gbtoolbox.misc as mt
import gbtoolbox.dft as dft
import numpy as np
import time
import matplotlib.pyplot as plt
from scipy import interpolate
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
print(torch.__version__)
from datetime import datetime
from torch.utils.data import Dataset, DataLoader, TensorDataset

ModuleNotFoundError: No module named 'gbtoolbox'

In [3]:
N = 80
mult = 10*2
span = 2
x = (np.random.rand(mult*N).reshape(-1,2)*2-1)*span/2.0
y = np.sin(2.1*1*np.pi*x[:,0])*np.sin(3.4*1*np.pi*x[:,1])
xt = (np.random.rand(mult*N*N).reshape(-1,2)*2-1)*span/2.0
yt = np.sin(2.1*1*np.pi*xt[:,0])*np.sin(3.4*1*np.pi*xt[:,1])

NameError: name 'np' is not defined

In [4]:
#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('mps')

NameError: name 'torch' is not defined

In [5]:
print(torch.backends.cudnn.is_available())
print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())

NameError: name 'torch' is not defined

tensor_x = torch.Tensor(x) # transform to torch tensor
tensor_y = torch.Tensor(y.reshape(-1,1))
MM=45000
RR=15000

class ShallowRegressor2(nn.Module):
    def __init__(self,SS):
        super(ShallowRegressor2,self).__init__()
        self.fc1 = nn.Linear(2,SS)
        self.fc2 = nn.Linear(SS,1)     
      
        
    def forward(self,x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model2 = ShallowRegressor2(MM)
model2 = model2.to(device)
model2.eval()
tensor_x = tensor_x.to(device)
tensor_y = tensor_y.to(device)
criterion = nn.MSELoss()  # mean square error
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model2.parameters()),lr=0.0001)

In [9]:
from collections import deque

In [10]:

class ShallowRegressor(nn.Module):
    def __init__(self,num_features):
        super(ShallowRegressor,self).__init__()
        self.fc1 = nn.Linear(num_features,MM)
        self.fc2 = nn.Linear(MM,1)     
        #self.fc1.weight = torch.nn.Parameter(torch.Tensor(wh.T))
        #self.fc1.bias = torch.nn.Parameter(torch.Tensor(tv*zv))      
        #self.fc2.bias = torch.nn.Parameter(torch.Tensor(np.array([my])))
        #self.fc2.weight = torch.nn.Parameter(torch.Tensor((sne)*(sv).reshape(-1,1).T/MM))
      
        
    def forward(self,x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

NameError: name 'optimizer' is not defined

In [11]:
pytorch_model = ShallowRegressor(2)
w0=pytorch_model.fc1.weight.detach().cpu().numpy()
w1=pytorch_model.fc2.weight.detach().cpu().numpy()
b0=pytorch_model.fc1.bias.detach().cpu().numpy()
b1=pytorch_model.fc2.bias.detach().cpu().numpy()
print(w0.shape)
print(b1.shape)

NameError: name 'model2' is not defined

In [12]:
np.random.seed(0)
model=gbnn.NeuralNet(2,MM)
model.barron_init(2,w0,b0.reshape(-1,1),w1,b1.reshape(1,1))
print(model.params)
model.barron_init(2,(wh.T),(tv*zv).reshape(-1,1),((1000)*(sv).reshape(-1,1).T/MM),np.array([[0]]))
print(model.params)

NameError: name 'model2' is not defined

In [ ]:
train_loss = []
val_loss = []
bestmodel = model.params
check_output, __ = model.forward_pass(x.T,model.params,'regression')
bestcost = gbnn.NeuralNet.cost(y.reshape(-1,1).T, check_output[0], "regression")
loss_history = deque(maxlen=10 + 1)
for epoch in range(4000):
    tcosts = model.train(x.T, y.reshape(-1,1).T, epoch_num=1, batch_size=64,cost_fun='regression',lambd=0.01)
    train_loss.append(tcosts[0])
    #check_output, __ = model.forward_pass(X_val.T,model.params,'cross_entropy')
    #costs = gbnn.NeuralNet.cost(y_val.reshape(-1,1).T, check_output[0], "cross_entropy")
    #val_loss.append(tcosts[0])
    if tcosts[0]<bestcost:
        bestcost=tcosts[0]
        bestmodel=model.params
    loss_history.append(tcosts[0])
    if len(loss_history)>10:
        if loss_history.popleft() < min(loss_history):
            break
print(model)
print(epoch)